<a href="https://colab.research.google.com/github/Ishany0/PM2.5_AirQuality/blob/main/Using_sequential.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping

In [48]:
PSRA_data = pd.read_csv('/content/PRSA_data_2010.1.1-2014.12.31.csv')
df = pd.DataFrame(PSRA_data)

df = df.dropna()
encoded_df = pd.get_dummies(df , columns = ['cbwd'] , drop_first = True)

In [49]:
feature_cols = ['DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir', 'month', 'hour'] + \
               [c for c in encoded_df.columns if c.startswith('cbwd_')]
X = encoded_df[feature_cols]
Y = pd.Series(encoded_df['pm2.5'])
print(df.columns.tolist())
print(X.isna().sum())

X_train , X_test , Y_train , Y_test = train_test_split(X , Y , test_size = 0.2 , random_state = 42)

['No', 'year', 'month', 'day', 'hour', 'pm2.5', 'DEWP', 'TEMP', 'PRES', 'cbwd', 'Iws', 'Is', 'Ir']
DEWP       0
TEMP       0
PRES       0
Iws        0
Is         0
Ir         0
month      0
hour       0
cbwd_NW    0
cbwd_SE    0
cbwd_cv    0
dtype: int64


In [51]:
model = Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1),
])

In [52]:
model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae'])

In [53]:
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_scaled, Y_train, epochs=200, validation_split=0.2, callbacks=[es])


Epoch 1/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 6826.1606 - mae: 58.0809 - val_loss: 5637.6733 - val_mae: 53.4026
Epoch 2/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 5174.9678 - mae: 50.4281 - val_loss: 4948.8105 - val_mae: 49.6357
Epoch 3/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 4726.7476 - mae: 48.4494 - val_loss: 4768.1348 - val_mae: 48.4183
Epoch 4/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 4503.0142 - mae: 47.1077 - val_loss: 4531.3086 - val_mae: 46.6657
Epoch 5/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 4345.9766 - mae: 46.0529 - val_loss: 4400.4351 - val_mae: 47.3145
Epoch 6/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4256.3384 - mae: 45.5292 - val_loss: 4282.0635 - val_mae: 44.9320
Epoch 7/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4168.6938 - mae: 44.7274 - val_loss: 4211.0034 - val_mae: 45.1171
Epoch 8/200
836/836 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4121.6992 - mae: 44.4093 - val_loss: 4203.3364 - v

In [54]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_r2 = r2_score(Y_train, y_train_pred)
test_r2 = r2_score(Y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(Y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(Y_test, y_test_pred))

print(f"Train R²: {train_r2:.4f}  |  Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}  |  Test RMSE: {test_rmse:.4f}")

1044/1044 ━━━━━━━━━━━━━━━━━━━━ 1s 793us/step
261/261 ━━━━━━━━━━━━━━━━━━━━ 0s 806us/step
Train R²: -167281.9035  |  Test R²: -159057.2709
Train RMSE: 37459.9034  |  Test RMSE: 37437.2053


In [58]:
train_r2 = r2_score(Y_train, y_train_pred)
test_r2 = r2_score(Y_test, y_test_pred)
train_rmse = root_mean_squared_error(Y_train, y_train_pred)
test_rmse = root_mean_squared_error(Y_test, y_test_pred)
print(f"Train R²: {train_r2:.4f}  |  Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}  |  Test RMSE: {test_rmse:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Train', 'Test'], [train_r2, test_r2], color=['steelblue', 'orange'])
axes[0].set_title('R² Score')
axes[0].set_ylim(0, 1)
for i, v in enumerate([train_r2, test_r2]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha='center')

axes[1].bar(['Train', 'Test'], [train_rmse, test_rmse], color=['steelblue', 'orange'])
axes[1].set_title('RMSE')
for i, v in enumerate([train_rmse, test_rmse]):
    axes[1].text(i, v + max(train_rmse, test_rmse) * 0.02, f"{v:.2f}", ha='center')

plt.tight_layout()
plt.show()

Train R²: -167281.9035  |  Test R²: -159057.2709
Train RMSE: 37459.9034  |  Test RMSE: 37437.2053


/tmp/ipykernel_4476/2945910215.py:21: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


ValueError: Image size of 826x51523173 pixels is too large. It must be less than 2^23 in each direction.

<Figure size 1000x400 with 2 Axes>